# Streamlit Football Object Detection App in Colab

This notebook runs a Streamlit app for football QB tracking using Roboflow inference.

In [1]:
# Install required packages
!pip install streamlit inference[transformers] roboflow numpy pyngrok
!pip install --upgrade opencv-contrib-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.7/105.7 kB 4.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 4.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.2 MB/s eta 0:00:00
  Prepa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 MB 9.1 MB/s eta 0:00:00:00:0100:01m
  Attempting uninstall: opencv-contrib-python
    Found existing installation: opencv-contrib-python 4.10.0.84
    Uninstalling opencv-contrib-python-4.10.0.84:
      Successfully uninstalled opencv-contrib-python-4.10.0.84
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inference 1.2.5 requires opencv-contrib-python<=4.10.0.84,>=4.8.1.78, but you have opencv-contrib-python 4.13.0.92 which is incompatible.


In [2]:
# Create the Streamlit app code
app_code = '''
import streamlit as st
import cv2
import numpy as np
from inference import get_model
from roboflow import Roboflow
import os
import shutil
import subprocess
import tempfile

st.set_page_config(page_title="Tommy Roboflow Football Tracker", layout="wide")
st.title("Tommy Roboflow Football Tracker")

ROBOFLOW_WORKSPACE = "tommys-workspace-vmucs"
ROBOFLOW_PROJECT = "rishi-fohsb-fdsoi"
ROBOFLOW_VERSION = 1
ROBOFLOW_MODEL_ID = f"{ROBOFLOW_PROJECT}/{ROBOFLOW_VERSION}"
MAX_DETECTION_FRAMES = 90
MIN_BBOX_AREA = 500


st.sidebar.header("Configuration")
api_key = st.sidebar.text_input("Roboflow API Key", type="password")
track_mode = st.sidebar.radio("Track mode", ["Track QB", "Track Specific ID"])
track_id = ""
if track_mode == "Track Specific ID":
    preview_detections = st.session_state.get("preview_detections", [])
    preview_ids = [str(det["id"]) for det in preview_detections]
    if preview_ids:
        track_id = st.sidebar.selectbox("Target ID", preview_ids)
    else:
        track_id = st.sidebar.text_input("Target ID (show ID preview first)")

video_source = st.sidebar.radio("Video source", ["Upload video", "Choose sideline video"])
sideline_videos_dir = "/content/drive/MyDrive/Unstructured Group Folder/NFL 1st and Future - Impact Detection - Data/nfl-impact-detection (Unzipped Files)/train/"

if video_source == "Choose sideline video":
    if os.path.exists(sideline_videos_dir):
        video_files = [f for f in os.listdir(sideline_videos_dir) if f.endswith('.mp4') and 'Sideline' in f]
        if video_files:
            selected_video = st.sidebar.selectbox("Select sideline video", video_files)
            video_path = os.path.join(sideline_videos_dir, selected_video)
        else:
            st.sidebar.error("No sideline videos found in the directory.")
            video_path = None
    else:
        st.sidebar.error("Sideline videos directory not found. Please check the path.")
        video_path = None
else:
    uploaded_file = st.sidebar.file_uploader("Upload video", type=["mp4", "avi", "mov"])
    if uploaded_file is not None:
        with tempfile.NamedTemporaryFile(delete=False, suffix='.mp4') as tmp_file:
            tmp_file.write(uploaded_file.read())
            video_path = tmp_file.name
    else:
        video_path = None

confidence_threshold = st.sidebar.slider("Confidence threshold", 0.0, 1.0, 0.5)
show_frame = st.sidebar.button("Show labeled preview")
run_tracking = st.sidebar.button("Run video tracking")


def create_tracker():
    try:
        return cv2.TrackerCSRT_create()
    except AttributeError:
        return cv2.legacy.TrackerCSRT_create()


def get_frame(video_path, frame_index=0):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames > frame_index:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None


def get_class_colors(detections):
    class_colors = {}
    palette = [
        (255, 0, 0), (0, 255, 0), (0, 0, 255),
        (255, 165, 0), (128, 0, 128), (0, 255, 255)
    ]
    for det in detections:
        cls = det["class"]
        if cls not in class_colors:
            class_colors[cls] = palette[len(class_colors) % len(palette)]
    return class_colors


def annotate_detections(frame, detections):
    display = frame.copy()
    class_colors = get_class_colors(detections)
    for det in detections:
        cls = det["class"]
        color = class_colors[cls]
        x, y, w, h = det["bbox"]
        cv2.rectangle(display, (x, y), (x + w, y + h), color, 2)
        label = f"ID:{det['id']} {cls} {det['conf']:.2f}"
        cv2.putText(display, label, (x, max(20, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return display, class_colors


def build_detections(results, min_conf):
    detections = []
    for i, pred in enumerate(results.predictions):
        if pred.confidence < min_conf:
            continue
        x = max(0, int(pred.x - pred.width / 2))
        y = max(0, int(pred.y - pred.height / 2))
        w = int(pred.width)
        h = int(pred.height)
        if w * h < MIN_BBOX_AREA:
            continue
        detections.append({
            "id": i,
            "class": pred.class_name,
            "conf": pred.confidence,
            "bbox": (x, y, w, h),
        })
    return detections


def load_model(api_key):
    rf = Roboflow(api_key=api_key)
    rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT).version(ROBOFLOW_VERSION)
    model = get_model(model_id=ROBOFLOW_MODEL_ID, api_key=rf.api_key)
    return model


def select_target(detections, track_mode, track_id):
    if track_mode == "Track QB":
        qb_candidates = [d for d in detections if d["class"].lower() == "qb"]
        return max(qb_candidates, key=lambda d: d["conf"]) if qb_candidates else None

    if track_id is None or track_id == "":
        return None
    match = next((d for d in detections if str(d["id"]) == str(track_id)), None)
    return match


def show_preview_for_choice(model, video_path, track_mode, confidence_threshold):
    frame = get_frame(video_path, frame_index=0)
    if frame is None:
        st.error("Could not open video file or read frame.")
        return None
    results = model.infer(frame)[0]
    detections = build_detections(results, confidence_threshold)
    show_ids = track_mode == "Track Specific ID"
    annotated_bgr, class_colors = annotate_detections(frame, detections)
    annotated = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
    caption = "ID preview of the first frame" if show_ids else "Labeled position preview of the first frame"
    st.image(annotated, channels="RGB", caption=caption)
    show_detected_table(detections, show_ids=show_ids)
    if show_ids:
        st.write("Use the ID list above to select a Target ID for specific tracking.")
    st.session_state["preview_detections"] = detections
    st.session_state["class_colors"] = class_colors
    return detections


def show_detected_table(detections, show_ids=False):
    if not detections:
        st.warning("No detections found on the preview frame.")
        return
    if show_ids:
        df = [
            {
                "ID": det["id"],
                "Class": det["class"],
                "Confidence": round(det["conf"], 3),
                "BBox": det["bbox"],
            }
            for det in detections
        ]
        st.write("### Detected players")
        st.code("\\n".join(
            f"ID {det['id']:2d} — {det['class']:15s} conf: {det['conf']:.2f}  bbox: {det['bbox']}"
            for det in detections
        ))
    else:
        df = [
            {
                "Position": det["class"],
                "Confidence": round(det["conf"], 3),
                "BBox": det["bbox"],
            }
            for det in detections
        ]
        st.write("### Labeled positions on preview frame")
    st.table(df)


def draw_target(frame, target, bbox=None, class_colors=None):
    x, y, w, h = [int(v) for v in (bbox or target["bbox"])]
    if class_colors is None:
        class_colors = get_class_colors([target])
    color = class_colors.get(target["class"], (255, 0, 0))
    label = f"ID:{target['id']} {target['class']}"
    cv2.rectangle(frame, (x, y), (x + w, y + h), color, 3)
    cv2.putText(frame, label, (x, max(25, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
    return frame


def browser_ready_video(input_path):
    ffmpeg = shutil.which("ffmpeg")
    if not ffmpeg:
        return input_path
    output_path = tempfile.mktemp(suffix=".mp4")
    cmd = [
        ffmpeg,
        "-y",
        "-i", input_path,
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-an",
        output_path,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return output_path if result.returncode == 0 and os.path.exists(output_path) else input_path


if api_key and video_path:
    model = load_model(api_key)

    if show_frame:
        frame = get_frame(video_path, frame_index=0)
        if frame is None:
            st.error("Could not open video file or read frame.")
        else:
            results = model.infer(frame)[0]
            detections = build_detections(results, confidence_threshold)
            show_ids = track_mode == "Track Specific ID"
            annotated_bgr, class_colors = annotate_detections(frame, detections)
            annotated = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
            caption = "ID preview of the first frame" if show_ids else "Labeled position preview of the first frame"
            st.image(annotated, channels="RGB", caption=caption)
            show_detected_table(detections, show_ids=show_ids)
            if show_ids:
                st.write("Use the ID list above to select a Target ID for specific tracking.")
            st.session_state["preview_detections"] = detections
            st.session_state["class_colors"] = class_colors

    if run_tracking:
        if track_mode == "Track Specific ID" and not track_id:
            st.error("Enter a Target ID before running specific ID tracking.")
            show_preview_for_choice(model, video_path, track_mode, confidence_threshold)
        else:
            frame = get_frame(video_path, frame_index=0)
            if frame is None:
                st.error("Could not open video file or read frame.")
            else:
                results = model.infer(frame)[0]
                detections = build_detections(results, confidence_threshold)
                class_colors = get_class_colors(detections)
                target = select_target(detections, track_mode, track_id)
                if target is None:
                    st.error("Could not identify the requested target in the preview frame.")
                else:
                    cap = cv2.VideoCapture(video_path)
                    if not cap.isOpened():
                        st.error("Could not open video file.")
                    else:
                        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                        fps = cap.get(cv2.CAP_PROP_FPS) or 30
                        raw_output_path = tempfile.mktemp(suffix='.mp4')
                        out = cv2.VideoWriter(raw_output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

                        ret, first_frame = cap.read()
                        if not ret:
                            st.error("Could not read video frames.")
                        else:
                            if not out.isOpened():
                                st.error("Could not create output video file.")
                                cap.release()
                            else:
                                tracker = create_tracker()
                                tracker.init(first_frame, target["bbox"])
                                out.write(draw_target(first_frame.copy(), target, class_colors=class_colors))

                                while True:
                                    ret, frame = cap.read()
                                    if not ret:
                                        break
                                    success, bbox = tracker.update(frame)
                                    if success:
                                        draw_target(frame, target, bbox, class_colors=class_colors)
                                    else:
                                        cv2.putText(frame, "Tracking lost", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
                                    out.write(frame)

                                cap.release()
                                out.release()
                                output_path = browser_ready_video(raw_output_path)
                                st.subheader("Tracked Video")
                                with open(output_path, "rb") as video_file:
                                    st.video(video_file.read())
                                st.success("Video tracking completed!")
else:
    if run_tracking or show_frame:
        st.error("Please provide a Roboflow API key and select/upload a video.")

st.markdown("""
## Instructions
1. Enter your Roboflow API key.
2. Upload a video or choose one from sideline videos.
3. Click "Show labeled preview" to see position labels for QB mode or ID labels for specific ID mode.
4. Select "Track QB" or "Track Specific ID".
5. If using specific ID, type the ID shown in the Detected players list.
6. Click "Run video tracking" to generate the tracked video.
""")
'''

# Write the app code to files
for app_file in ['tommy_streamlit_app.py', 'app.py']:
    with open(app_file, 'w') as f:
        f.write(app_code)

print("Streamlit app created as tommy_streamlit_app.py and app.py")

Streamlit app created as tommy_streamlit_app.py


In [3]:
# Mount Google Drive (if needed for videos)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Run the Streamlit app with ngrok for public access
from pyngrok import ngrok, conf
import os
import subprocess
import time



print("Cleaning up old ngrok sessions...")
try:
    ngrok.kill()
except Exception:
    pass

ngrok_token = "3Cmwmci4i9e7gWgUwF5FsRFoM9m_43KtgNzk5C4eb9M1jGAw4"
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
else:
    print("No NGROK_AUTHTOKEN secret found. Add one in Colab secrets if ngrok requires auth.")

print("Starting Streamlit...")
process = subprocess.Popen([
    'streamlit', 'run', 'tommy_streamlit_app.py',
    '--server.port', '8501',
    '--server.address', '0.0.0.0',
    '--server.headless', 'true',
])

print("Waiting 5 seconds for Streamlit to initialize...")
time.sleep(5)

print("Setting up ngrok tunnel...")
try:
    public_url = ngrok.connect(8501, bind_tls=True)
    print(f"Streamlit app is available at: {public_url}")
    print("Both Streamlit and ngrok are running. Open the URL above in a new browser tab.")

    while True:
        time.sleep(1)
except Exception as exc:
    process.terminate()
    ngrok.kill()
    print("ngrok failed before the app could be shared.")
    print("If this happens again, restart the Colab runtime and confirm your NGROK_AUTHTOKEN secret is valid.")
    raise exc
except KeyboardInterrupt:
    process.terminate()
    ngrok.kill()

Cleaning up old ngrok sessions...


Starting Streamlit...                                                                               
Waiting 5 seconds for Streamlit to initialize...
Setting up ngrok tunnel...
Streamlit app is available at: NgrokTunnel: "https://timing-domain-gestation.ngrok-free.dev" -> "http://localhost:8501"
Both Streamlit and ngrok are running. Open the URL above in a new browser tab.
